In [2]:
import pandas as pd
from time import sleep
import datetime
import os
from bs4 import BeautifulSoup
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep

print("Running HU CBH Web Scraping Tool v.1.0")
regulatorName = 'HU CBH'
#scriptfolder=os.path.dirname(os.path.abspath(__file__))
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

os.chdir(scriptfolder)
now=datetime.datetime.now()
filename= 'HU CBH data {}.xlsx'.format(str(now).replace(":",".")[:-7])


#The following Regcodes are excluded from the reglist: 'HU CBH 4', 'HU CBH 18'


regdict= {'HU CBH 1': ['//*[@id="-6"]/div[2]', '//*[@id="-20"]/div[2]']
        , 'HU CBH 2': ['//*[@id="-6"]/div[2]', '//*[@id="-19"]/div[2]']
        , 'HU CBH 3': ['//*[@id="-8"]/div[2]', '//*[@id="-25"]/div[2]']
        , 'HU CBH 4': ['', '']
        , 'HU CBH 5': ['//*[@id="-8"]/div[2]', '//*[@id="132"]/div[2]']
        , 'HU CBH 6': ['//*[@id="-8"]/div[2]', '//*[@id="-24"]/div[2]']
        , 'HU CBH 7': ['//*[@id="-1"]/div[2]', '//*[@id="-10"]/div[2]']
        , 'HU CBH 8': ['//*[@id="-1"]/div[2]', '//*[@id="-9"]/div[2]']
        , 'HU CBH 9': ['//*[@id="-7"]/div[2]', '//*[@id="39"]/div[2]']
        , 'HU CBH 10': ['//*[@id="-7"]/div[2]', '//*[@id="-23"]/div[2]']
        , 'HU CBH 11': ['//*[@id="-5"]/div[2]', '//*[@id="-17"]/div[2]']
        , 'HU CBH 12': ['//*[@id="-5"]/div[2]', '//*[@id="-18"]/div[2]']
        , 'HU CBH 13': ['//*[@id="-3"]/div[2]', '//*[@id="-13"]/div[2]']
        , 'HU CBH 14': ['//*[@id="-3"]/div[2]', '//*[@id="-15"]/div[2]']
        , 'HU CBH 15': ['//*[@id="-3"]/div[2]', '//*[@id="-14"]/div[2]']
        , 'HU CBH 16': ['//*[@id="-3"]/div[2]', '//*[@id="-12"]/div[2]']
        , 'HU CBH 17': ['//*[@id="-4"]/div[2]', '//*[@id="-16"]/div[2]']
        , 'HU CBH 18': ['//*[@id="-4"]/div[2]', '']}

driver = webdriver.Chrome()
driver.maximize_window()
# Navigate to the Chrome Web Store URL of the desired extension
extension_url = 'https://chromewebstore.google.com/detail/rektcaptcha-recaptcha-sol/bbdhfoclddncoaomddgkaaphcnddbpdh?hl=en-US&utm_source=ext_sidebar'
driver.get(extension_url)
driver.maximize_window()
sleep(5)

# Automate the "Add to Chrome" process
try:
    add_button = driver.find_element(By.XPATH, '//button[span[contains(text(),"Add to Chrome")]]')
    add_button.click()
    sleep(8)
    pyautogui.press('tab', presses=1)
    sleep(1)
    pyautogui.press('enter') # Confirm "Add Extension"
    sleep(2)
except Exception as e:
    print(f"Error occurred: {e}")
    # Close the browser
    sleep(5)


catlist=['Name', 'Previous name', 'Administrative address', 'Type of institution', 'Registration number', 'Registry court/court number', 
        'Website address', 'Location of publication', 'Legal status', '[EN]Közérdeklődésre számon tartott hitelintézet']


xname=[]
xpname=[]
xadmadd=[]
xtypeinst=[]
xregnum=[]
xregcourt=[]
xweb=[]
xlocpub=[]
xlegsta=[]
xen=[]
xauth=[]
clinks=[]
reglist = []

for reg in regdict:
    print('Working with {}.'.format(reg))

    driver.get('https://intezmenykereso.mnb.hu/en/Home/Index')
    driver.find_element(By.XPATH, '/html/body/div[3]/div[6]').click()
    sleep(1)
    driver.find_element(By.XPATH, '//*[@id="complex-search-panel"]/div[1]').click()
    driver.find_element(By.XPATH, regdict[reg][0]).click()
    sleep(2)
    try:
        driver.find_element(By.XPATH, regdict[reg][1]).click()
    except:
        sleep(2)
        driver.find_element(By.XPATH, regdict[reg][1]).click()
    sleep(5)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    sleep(2)
    button_ = driver.find_element(By.XPATH, '//*[@id="complex-inst-search-button"]')
    sleep(3)
    button_.click()
    sleep(2)
    # while True:

    #     captcha=input('Solved Captcha? (Y/N): ')

    #     if captcha=='Y':

    #         break

    #     else:

    #         print('Type Y when the captcha has been manually solved.')



    #driver.switch_to.default_content()

    source=driver.page_source

    tempstr=BeautifulSoup(source, 'html.parser')

    count=int(tempstr.find('text', {'id':'count'}).text)

    tcount=int(tempstr.find('text', {'id':'total-count'}).text)

    while count<tcount:

        driver.execute_script("var scrollingElement = (document.scrollingElement || document.body);scrollingElement.scrollTop = scrollingElement.scrollHeight;")

        sleep(0.1)

        driver.find_element(By.XPATH, '//body').send_keys(Keys.CONTROL+Keys.END)

        try:

            driver.find_element(By.XPATH, '//*[@id="result-table"]/div[5]/div[1]/input').click()

        except:

            sleep(1)

            source=driver.page_source

            tempstr=BeautifulSoup(source, 'html.parser')

            count=int(tempstr.find('text', {'id':'count'}).text)

    source=driver.page_source

    tempstr=BeautifulSoup(source, 'html.parser')

    rows=tempstr.find_all('div',  {'class':'result-table-row'})

    for row in rows:

        auth=row.find_all("div",  {"class":"result-table-cell"})[3]

        auth=auth.find('input', value= True)['value'].strip()

        if 'Not' not in auth:

            auth=auth.split(' ')[0].strip()

        xauth.append(auth)

        row=row.find_all("div",  {"class":"result-table-cell"})[5]

        lid=''

        lid=row.find('input', lid=True)['lid']

        clinks.append(lid)

    for link in range(len(clinks)):

        print('Working with firm {} of {} firms. (Reg code: {})'.format(link+1, len(clinks), reg))

        tempdict={}

        page='https://intezmenykereso.mnb.hu/en/Details/Index?LId='+clinks[link]+'&EntityType=Institute&expandAccordions=IntezmenyAlapadatok'

        driver.get(page)

        sleep(0.25)

        soup=BeautifulSoup(driver.page_source, 'html.parser')

        datadiv=soup.find('div', {"id":"details-list"})

        if datadiv!=None:

            data=datadiv.find_all('div',  {'class':"result-table-row"})

        else:

            data=[]

            print('datadiv is None.')

        for ro in data:

            ro=ro.find_all("div",  {"class":"result-table-cell"})

            tempdict[ro[0].text.strip()]=ro[1].text.strip()

        for ca in catlist:

            if ca not in tempdict:

                tempdict[ca]=''

            else:

                pass

        xname.append(tempdict[catlist[0]])

        xpname.append(tempdict[catlist[1]])

        xadmadd.append(tempdict[catlist[2]])

        xtypeinst.append(tempdict[catlist[3]])

        xregnum.append(tempdict[catlist[4]])

        xregcourt.append(tempdict[catlist[5]])

        xweb.append(tempdict[catlist[6]])

        xlocpub.append(tempdict[catlist[7]])

        xlegsta.append(tempdict[catlist[8]])

        xen.append(tempdict[catlist[9]])

        reglist.append(reg)
                        

df= pd.DataFrame({'Reg List': reglist, 'Name': xname, 'Previous name': xpname, 'Administrative address': xadmadd,'Authorization': xauth, 'Type of institution': xtypeinst, 'Registration number': xregnum, 'Registry court/court number': xregcourt, 'Website address': xweb, 'Location of publication': xlocpub, 'Legal status': xlegsta, '[EN]Közérdeklődésre számon tartott hitelintézet': xen})
writer = ExcelWriter(filename)
df.to_excel(writer, reg)


writer.save()
writer.close()

sleep(3)


driver.quit()





    
    

Running HU CBH Web Scraping Tool v.1.0
Error occurred: name 'pyautogui' is not defined
Working with HU CBH 1.


ElementClickInterceptedException: Message: element click intercepted: Element is not clickable at point (632, 654)
  (Session info: chrome=142.0.7444.176); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff79e82a235
	0x7ff79e582630
	0x7ff79e3116dd
	0x7ff79e3724c9
	0x7ff79e36fe4e
	0x7ff79e36cd71
	0x7ff79e36bc10
	0x7ff79e35d3a8
	0x7ff79e392b0a
	0x7ff79e35cc36
	0x7ff79e3bbaba
	0x7ff79e35b0ed
	0x7ff79e35bf63
	0x7ff79e855d60
	0x7ff79e84fe8a
	0x7ff79e871005
	0x7ff79e59d71e
	0x7ff79e5a4e1f
	0x7ff79e58b7c4
	0x7ff79e58b97f
	0x7ff79e5718e8
	0x7ffb25ae259d
	0x7ffb27aaaf78


In [ ]:
import pyautogui
import pytesseract
from PIL import ImageGrab
import time
import pandas as pd
from time import sleep
import datetime
import os
from bs4 import BeautifulSoup
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
driver = webdriver.Chrome()
driver.maximize_window()


# Navigate to the Chrome Web Store URL of the desired extension
extension_url = 'https://chromewebstore.google.com/detail/rektcaptcha-recaptcha-sol/bbdhfoclddncoaomddgkaaphcnddbpdh?hl=en-US&utm_source=ext_sidebar'
driver.get(extension_url)
driver.maximize_window()
time.sleep(5)

# Automate the "Add to Chrome" process
try:
    add_button = driver.find_element(By.XPATH, '//button[span[contains(text(),"Add to Chrome")]]')
    add_button.click()
    time.sleep(8)
    pyautogui.press('tab', presses=1)
    time.sleep(1)
    pyautogui.press('enter') # Confirm "Add Extension"
    time.sleep(2)
    # jump straight to Chrome’s keyboard shortcut settings
    driver.get("chrome://extensions/shortcuts")
    time.sleep(5)
    shortcut_el = driver.execute_script("""
                const manager = document.querySelector('extensions-manager');
                const shortcutsPage = manager.shadowRoot.querySelector('extensions-shortcuts');
                const list = shortcutsPage.shadowRoot.querySelector('extensions-shortcut-list');
                const items = list.shadowRoot.querySelectorAll('extensions-shortcut-item');

                for (const item of items) {
                const title = item.shadowRoot.querySelector('#title').textContent.trim();
                if (title.includes('rektCaptcha')) {        // match your extension label
                    return item.shadowRoot.querySelector('#row-container #input');
                }
                }
                return null;
                """)

    if shortcut_el:
        driver.execute_script("arguments[0].removeAttribute('readonly');", shortcut_el)
        shortcut_el.click()
        shortcut_el.send_keys(Keys.CONTROL, "4")
    else:
        print("Shortcut input not found")
except Exception as e:
    print(f"Error occurred: {e}")
    # Close the browser
    time.sleep(5)
# Set the path to tesseract.exe if it's not in your PATH environment variable
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\Tesseract-OCR\tesseract.exe"

time.sleep(5) # Wait for 5 seconds to prepare
# Capture the entire screen
screenshot = ImageGrab.grab()
# Convert the image to string using pytesseract
text_data = pytesseract.image_to_data(screenshot, output_type=pytesseract.Output.DICT)
# The text you want to detect
target_text = "ext_sidebar"
# Loop through all detected text to find coordinates of the target text
for i, text in enumerate(text_data['text']):
   if target_text.lower() in text.lower():
       x = text_data['left'][i]
       y = text_data['top'][i]
       width = text_data['width'][i]
       height = text_data['height'][i]
       # Calculate the center of the text
       center_x = x + width // 2 +20
       center_y = y + height // 2 -20
       # Move the mouse to the center of the text and click
       pyautogui.moveTo(center_x, center_y)
       pyautogui.click()
       print(f"Clicked on text '{target_text}' at ({center_x}, {center_y})")
       break
else:
   print(f"Text '{target_text}' not found on the screen.")

Error occurred: Message: javascript error: Cannot read properties of null (reading 'shadowRoot')
  (Session info: chrome=142.0.7444.176)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7f918a235
	0x7ff7f8ee2630
	0x7ff7f8c716dd
	0x7ff7f8c794c8
	0x7ff7f8c7c8c2
	0x7ff7f8d1ce53
	0x7ff7f8cf2b0a
	0x7ff7f8d1baba
	0x7ff7f8cbb0ed
	0x7ff7f8cbbf63
	0x7ff7f91b5d60
	0x7ff7f91afe8a
	0x7ff7f91d1005
	0x7ff7f8efd71e
	0x7ff7f8f04e1f
	0x7ff7f8eeb7c4
	0x7ff7f8eeb97f
	0x7ff7f8ed18e8
	0x7ffb25ae259d
	0x7ffb27aaaf78

Text 'ext_sidebar' not found on the screen.


In [ ]:
for i, text in enumerate(text_data['text']):
   if target_text.lower() in text.lower():
       x = text_data['left'][i]
       y = text_data['top'][i]
       width = text_data['width'][i]
       height = text_data['height'][i]
       # Calculate the center of the text
       center_x = x + width // 2 
       center_y = y + height // 2 -20
       # Move the mouse to the center of the text and click
       pyautogui.moveTo(center_x, center_y)
       pyautogui.click()
       print(f"Clicked on text '{target_text}' at ({center_x}, {center_y})")